**Necessary imports**

In [1]:
import subprocess
import os
import pandas as pd

%load_ext autoreload
%autoreload 2

**Setting parameters**

In [3]:
simple_render_raw_data_dir = "/dlabdata1/tsoares/wikidumps/"
processed_data_dir = "/dlabdata1/boulenge/wikidumps_data/report_examples"
processes = 48
max_samples_per_lang = 100
langs = ['fr','es', 'en', 'de'] #sorted(set([filename.split("wiki")[0] for filename in os.listdir(simple_render_raw_data_dir) if "wiki" in filename]))
print(langs)
side_context_sizes = [5]  # Example context sizes
second_month = "20231001"
third_month = "20231101"

langs_to_remove = ["war", "ky", "nds-nl"]

for lang in langs_to_remove:
    if lang in langs:
        langs.remove(lang)



['fr', 'es', 'en', 'de']


**Defining data paths**

In [4]:
# Define directories
rev_files_dir = os.path.join(processed_data_dir, "revisions_files")
parsed_htmls_dir = os.path.join(processed_data_dir, "parsed_htmls")
contexts_splitted_data_dir = os.path.join(processed_data_dir, "contexts_splitted_data")
ready_to_score_data_dir = os.path.join(processed_data_dir, "ready_to_score_data")
scored_data_dir = os.path.join(processed_data_dir, "scored_data")
crawling_output_dir = os.path.join(processed_data_dir, "crawled_pages")

# Create necessary directories if they don't exist
os.makedirs(rev_files_dir, exist_ok=True)
os.makedirs(parsed_htmls_dir, exist_ok=True)
os.makedirs(contexts_splitted_data_dir, exist_ok=True)
os.makedirs(ready_to_score_data_dir, exist_ok=True)
os.makedirs(scored_data_dir, exist_ok=True)
os.makedirs(crawling_output_dir, exist_ok=True)

**Prepare files to be crawled and parsed**

In [5]:
for lang in langs:
    try:
        print(f"Processing language: {lang}")
        second_month_path = f"{simple_render_raw_data_dir}/{lang}wiki-NS0-{second_month}"
        third_month_path = f"{simple_render_raw_data_dir}/{lang}wiki-NS0-{third_month}"
        save_path = f"{rev_files_dir}/revisions_{lang}.parquet"
            
        subprocess.run([
            "python", "generate_revisions_file.py",
            "--versions_file", f"{second_month_path}/eval/link_versions",
            "--html_dir", f"{second_month_path}/eval/pages",
            "--redirect_1", f"{second_month_path}/processed_data/redirect_map.parquet",
            "--redirect_2", f"{third_month_path}/processed_data/redirect_map.parquet",
            "--mention_map", f"{second_month_path}/processed_data/mention_map.parquet",
            "--max_samples", str(max_samples_per_lang),
            "--save_path", save_path
        ])

        #revisions = pd.read_parquet(save_path)
        #display(revisions.head(5))
    except Exception as e:
        print(f"Error processing language {lang}: {e}")
        continue

Processing language: fr


Checking revisions: 100%|██████████| 26847/26847 [03:08<00:00, 142.29it/s]


Processing language: es


Checking revisions: 100%|██████████| 29970/29970 [04:21<00:00, 114.76it/s]


Processing language: en


Checking revisions: 100%|██████████| 194300/194300 [26:02<00:00, 124.36it/s] 


Processing language: de


Checking revisions: 100%|██████████| 49899/49899 [06:19<00:00, 131.54it/s]


**Crawling first versions only**

In [6]:
for lang in langs:
    print(f"Processing language: {lang}")
    subprocess.run([
        "python", "crawl.py",
        "--revisions_path", f"{rev_files_dir}/revisions_{lang}.parquet",
        "--output_dir", f"{crawling_output_dir}/{lang}",
        "--n_processes", str(processes),
        "--lang", lang,
        #"--crawl_second_versions", "False" (false by default)
    ])

Processing language: fr
Total number of pages to crawl: 100
Running crawling on 48 processes
Parallel multiprocess
Crawling finished, post processing...


Moving files: 100%|██████████| 100/100 [00:00<00:00, 122.78it/s]


Processing language: es
Total number of pages to crawl: 100
Running crawling on 48 processes
Parallel multiprocess
Crawling finished, post processing...


Moving files: 100%|██████████| 100/100 [00:00<00:00, 121.57it/s]


Processing language: en
Total number of pages to crawl: 100
Running crawling on 48 processes
Parallel multiprocess
Crawling finished, post processing...


Moving files: 100%|██████████| 100/100 [00:00<00:00, 171.74it/s]


Processing language: de
Total number of pages to crawl: 100
Running crawling on 48 processes
Parallel multiprocess
Crawling finished, post processing...


Moving files: 100%|██████████| 100/100 [00:00<00:00, 196.21it/s]


**Parsing of the htmls**

In [7]:
for lang in langs:
    print(f"Processing language: {lang}")
    second_month_path = f"{simple_render_raw_data_dir}/{lang}wiki-NS0-{second_month}"
    first_version_specific_html_dir = f"{crawling_output_dir}/{lang}"
    save_path = f"{rev_files_dir}/revisions_{lang}.parquet"
    output_dir = f"{parsed_htmls_dir}/{lang}_v2"
    first_version_specific_output_dir = f"{parsed_htmls_dir}/{lang}_v1"
        
    subprocess.run([
        "python", "parsing.py",
        "--revisions_path", save_path,        
        "--html_dir", f"{second_month_path}/eval/pages",
        "--first_versions_specific_html_dir", first_version_specific_html_dir,
        "--output_dir", output_dir,
        "--first_versions_specific_output_dir", first_version_specific_output_dir,
        "--n_processes", str(processes)
    ])

    parsed_htmls_paths = os.listdir(first_version_specific_output_dir)
    sentences_path = list(filter(lambda x: x.endswith("sentences.jsonl"), parsed_htmls_paths))[0]
    links_path = list(filter(lambda x: x.endswith("links.jsonl"), parsed_htmls_paths))[0]
    sentences_df = pd.read_json(f"{first_version_specific_output_dir}/{sentences_path}")
    links_df = pd.read_json(f"{first_version_specific_output_dir}/{links_path}")

    display(sentences_df.head(5))
    display(links_df.head(5))   

Processing language: fr
First version html dir: /dlabdata1/boulenge/wikidumps_data/report_examples/crawled_pages/fr
Running on 48 processes
Parallel multiprocess
Writing sentences and links v1
Time taken for v1: 28.909183979034424
Writing sentences and links v2
Time taken for v2: 29.635920524597168


,md5,id,text,startIdx,endIdx,section,locList
0,d28bdd7e7c802efc7ef6eae2f9f96b7f,0,L'histoire de l'Algérie dans l'Antiquité comme...,48258,48918,Lead,"[{'start': 48258, 'end': 48260}, {'start': 482..."
1,d4f8e54633d2fc3a89986150a39a6c02,1,Ceux-ci sont principalement d'abord ceux des G...,48919,49210,Lead,"[{'start': 48918, 'end': 48964}, {'start': 490..."
2,41384850963a4436de45fbec5c79ebd2,2,"Puis viennent les Romains, les Vandales ariens...",49211,49579,Lead,"[{'start': 49209, 'end': 49228}, {'start': 492..."
3,170bbaa28015006f6549493cd1542aeb,3,La conquête musulmane du Maghreb au VIIe&nbsp;...,49580,49917,Lead,"[{'start': 49578, 'end': 49582}, {'start': 496..."
4,02635068b0c39422f9500a34ac7c6c69,4,Les Gétules peuplaient certaines parties de l'...,50050,50448,Gétules,"[{'start': 50049, 'end': 50050}, {'start': 500..."


,link,startIdx,text,endIdx,section,sentid
0,file:///wiki/Bataille_d%27Uji_(1221),43165,Troisième bataille d'Uji,43263,Lead,2
1,file:///wiki/Japon,44185,Japon,44230,Lead,4
2,file:///wiki/%C3%89poque_de_Kamakura,44245,Époque de Kamakura,44334,Lead,4
3,file:///wiki/Insei_(syst%C3%A8me_de_gouvernement),44372,empereur retiré,44484,Lead,4
4,file:///wiki/Go-Toba,44485,Go-Toba,44536,Lead,4


Processing language: es
First version html dir: /dlabdata1/boulenge/wikidumps_data/report_examples/crawled_pages/es
Running on 48 processes
Parallel multiprocess
Writing sentences and links v1
Time taken for v1: 34.803826332092285
Writing sentences and links v2
Time taken for v2: 34.7550265789032


,md5,id,text,startIdx,endIdx,section,locList
0,11f8496f2868b04dd4b830743fab93d3,0,El Club Deportivo Voleibol Río Duero Soria se ...,52335,52867,Lead,"[{'start': 52334, 'end': 52335}, {'start': 523..."
1,8c80e360a6dfff8c19f15ab6f295924b,1,Desde la temporada 2013/14 el club cuenta con ...,52868,53477,Lead,"[{'start': 52866, 'end': 52867}, {'start': 528..."
2,5b45e5aa494f8d56687078e08e68218e,2,También colabora en la organización del torneo...,53478,53607,Lead,"[{'start': 53476, 'end': 53606}]"
3,7a644c30b64c7df5660131ae00022abf,3,En verano de 2013 tras el anuncio por parte de...,53711,54415,Historia,"[{'start': 53710, 'end': 53711}, {'start': 537..."
4,784f5889953cfdbbbfbb54bcb7755751,4,El club decidió apostar por una plantilla más ...,54509,55679,Historia,"[{'start': 54508, 'end': 54509}, {'start': 545..."


...

,md5,id,text,startIdx,endIdx,section,locList
0,f31b08662d253887d0b1aee6b6d16349,0,Football league season,41098,43499,Lead,"[{'start': 41098, 'end': 41120}]"
1,a9e9672b1e069a1532c806990d8e4405,1,The 2022–23 Men's Feldhockey-Bundesliga was th...,43500,43796,Lead,"[{'start': 43499, 'end': 43500}, {'start': 435..."
2,80d9fc322cacde5f916d2dc5b09c2988,2,The season began on 3 September 2022 and concl...,43797,43966,Lead,"[{'start': 43795, 'end': 43890}, {'start': 439..."
3,9f5066faec1ac7d548d18f3fdc14f9fa,3,Rot-Weiss Köln were the two-time defending cha...,43967,44380,Lead,"[{'start': 43965, 'end': 43966}, {'start': 440..."
4,e305948c3d3e671ee10f221536c04715,4,A total of 12 teams participate in the 2022–23...,56381,56458,Teams,"[{'start': 56380, 'end': 56381}, {'start': 563..."


,link,startIdx,text,endIdx,section,sentid
0,file:///wiki/Star_Wars:_The_Bad_Batch,48673,Star Wars: The Bad Batch,48775,Events,2
1,file:///wiki/Disney%2B,48793,Disney+,48846,Events,2
2,file:///wiki/Shion_Takeuchi,48982,Shion Takeuchi,49054,Events,3
3,file:///wiki/Inside_Job_(2021_TV_series),49095,Inside Job,49189,Events,3
4,file:///wiki/Netflix,49216,Netflix,49267,Events,3


Processing language: de
First version html dir: /dlabdata1/boulenge/wikidumps_data/report_examples/crawled_pages/de
Running on 48 processes
Parallel multiprocess
Writing sentences and links v1
Time taken for v1: 47.61888575553894
Writing sentences and links v2
Time taken for v2: 49.748169898986816


,md5,id,text,startIdx,endIdx,section,locList
0,bd15d87a56a014b73ed0555d6af56673,0,"Eine gesichtete Version dieser Seite, die am 26.",8576,8741,Lead,"[{'start': 8576, 'end': 8581}, {'start': 8691,..."
1,62f82b99f02d593e21a4d93e915b89e6,1,"August 2023 freigegeben wurde, basiert auf die...",8742,9260,Lead,"[{'start': 8741, 'end': 8753}, {'start': 8757,..."
2,79afeab88e1ebd59a017476928f83adb,2,Leben,10633,10702,Leben,"[{'start': 10631, 'end': 10633}, {'start': 106..."
3,8942865c0bff71f2dbdaeffbe6c96d12,3,Rosen kam nach eigener Aussage über das Skaten...,10703,10812,Leben,"[{'start': 10700, 'end': 10701}, {'start': 107..."
4,13e10a03159e7648b9beacde33af6269,4,Er studiert Englisch und Philosophie auf Lehramt.,10813,11022,Leben,"[{'start': 10811, 'end': 10824}, {'start': 108..."


,link,startIdx,text,endIdx,section,sentid
0,file:///wiki/5._Juni,9658,5. Juni,9709,Lead,2
1,file:///wiki/1905,9710,1905,9752,Lead,2
2,file:///wiki/London,9756,London,9804,Lead,2
3,file:///wiki/24._Mai,9808,24. Mai,9859,Lead,3
4,file:///wiki/1996,9860,1996,9902,Lead,3


**Preprocessing**

In [8]:
for lang in langs:
    for side_context_size in side_context_sizes:
        print(f"Processing language: {lang} with side context size {side_context_size}")
        second_month_path = f"{simple_render_raw_data_dir}/{lang}wiki-NS0-{second_month}"
        third_month_path = f"{simple_render_raw_data_dir}/{lang}wiki-NS0-{third_month}"
        save_path = f"{rev_files_dir}/revisions_{lang}.parquet"
        output_dir = f"{contexts_splitted_data_dir}/{lang}/context_size_{side_context_size}"
        
        subprocess.run([
            "python", "modular_preprocessing.py",
            "--revisions_path", save_path,
            "--parsed_html_dir", f"{parsed_htmls_dir}/{lang}_v2",
            "--first_versions_specific_parsed_html_dir", f"{parsed_htmls_dir}/{lang}_v1",
            "--redirect_1", f"{second_month_path}/processed_data/redirect_map.parquet",
            "--redirect_2", f"{third_month_path}/processed_data/redirect_map.parquet",
            "--mention_map", f"{second_month_path}/processed_data/mention_map.parquet",
            "--output_dir", output_dir,
            "--side_contexts_size", str(side_context_size),
            "--n_processes", "1" #str(processes)
        ])

        #preprocessed_data_path = os.path.join(output_dir, f'test_data.parquet')

        #preprocessed_data = pd.read_parquet(preprocessed_data_path)
        #display(preprocessed_data.head(5))

Processing language: fr with side context size 5
Running on 1 processes


100%|██████████| 223/223 [00:00<00:00, 15368.04it/s]


The length of the output is 223
Processing language: es with side context size 5
Running on 1 processes


 96%|█████████▌| 96/100 [00:10<00:00, 20.63it/s]

No sentences found in 1
10698004_154891539_sentences.jsonl
The length of the output is 290


...

100%|██████████| 686/686 [00:00<00:00, 31653.38it/s]


The length of the output is 686
Processing language: de with side context size 5
Running on 1 processes


  0%|          | 0/903 [00:00<?, ?it/s]

The length of the output is 903


100%|██████████| 903/903 [00:00<00:00, 15837.56it/s]


**Merging**

In [9]:
test_data_dir = contexts_splitted_data_dir
output_dir = ready_to_score_data_dir

subprocess.run([
    "python", "modular_merge_data.py",
    "--test_data_dir", test_data_dir,
    "--raw_data_dir", simple_render_raw_data_dir,
    "--langs", ",".join(langs),
    "--side_contexts_sizes", ",".join(map(str, side_context_sizes)),
    "--output_dir", output_dir,
    "--ignore_warnings"
])

first_lang = langs[0]
first_context_size = str(side_context_sizes[0])
scored_files_example = os.listdir(output_dir)
scored_files_example = list(filter(lambda x: (first_lang in x and first_context_size in x), scored_files_example))
final_input_data_path = os.path.join(output_dir, scored_files_example[0])

final_input_data_df = pd.read_parquet(final_input_data_path)
display(final_input_data_df.head(5))

Saving mention maps
Processing language fr (1/4)
Finished processing language fr
Processing language es (2/4)
Finished processing language es
Processing language en (3/4)
Finished processing language en
Processing language de (4/4)
Finished processing language de
Processing language fr (1/4)
Loaded pages
Loaded redirects


100%|██████████| 2515482/2515482 [00:03<00:00, 768739.07it/s]


Processing context size 5
Loaded links for context_size_5
Originally, there are 223 links for language fr
Removed 0 links because of empty contexts
Removed 38 links because target title is not in source titles
Removed 25 links because of missing section
After cleaning, there are 160 links for language fr
Finished processing language fr
Processing language es (2/4)
Loaded pages
Loaded redirects


100%|██████████| 1829995/1829995 [00:02<00:00, 793300.76it/s]


Processing context size 5
Loaded links for context_size_5
Originally, there are 290 links for language es
Removed 0 links because of empty contexts
Removed 37 links because target title is not in source titles
Removed 84 links because of missing section
After cleaning, there are 169 links for language es
Finished processing language es
Processing language en (3/4)
Loaded pages
Loaded redirects


100%|██████████| 6708565/6708565 [00:09<00:00, 697529.78it/s]


Processing context size 5
Loaded links for context_size_5
Originally, there are 686 links for language en
Removed 0 links because of empty contexts
Removed 92 links because target title is not in source titles
Removed 403 links because of missing section
After cleaning, there are 191 links for language en
Finished processing language en
Processing language de (4/4)
Loaded pages
Loaded redirects


100%|██████████| 2826417/2826417 [00:03<00:00, 796683.93it/s]


Processing context size 5
Loaded links for context_size_5
Originally, there are 903 links for language de
Removed 0 links because of empty contexts
Removed 203 links because target title is not in source titles
Removed 467 links because of missing section
After cleaning, there are 233 links for language de
Finished processing language de


,source_title,source_ID,target_title,context,insert_indices,moved_insertion_marker,link_sentence,link_text,section,mention_present,source_lead,first_version,second_version,direct_match,missing_category,all_contexts,target_lead
0,Artillerie,57485,Arbal%C3%A8te,None,[21],None,L’arbalète et la baliste sont apparues pendant...,arbalète,Histoire,None,On appelle artillerie l'ensemble des armes col...,208533468,208533787,None,missing_mention_with_sentence_modifications,[{'context': 'On appelle artillerie l'ensemble...,"L'arbalète est une arme de trait, constituée d..."
1,Artillerie,57485,Baliste,None,[21],None,L’arbalète et la baliste sont apparues pendant...,baliste,Histoire,None,On appelle artillerie l'ensemble des armes col...,208533468,208533787,None,missing_mention_with_sentence_modifications,[{'context': 'On appelle artillerie l'ensemble...,La baliste était un engin de siège développé à...
2,Artillerie,57485,Mausol%C3%A9e_de_l%27empereur_Qin,None,[24],None,Des exemplaires de ces «&nbsp;armes du diable&...,armée de terre cuite,Histoire,None,On appelle artillerie l'ensemble des armes col...,208533468,208533787,None,missing_mention_with_sentence_modifications,[{'context': 'On appelle artillerie l'ensemble...,Le mausolée de l'empereur Qin se trouve à prox...
3,Artillerie,57485,Qin_Shi_Huang,None,[24],None,Des exemplaires de ces «&nbsp;armes du diable&...,Qin shi Huang,Histoire,None,On appelle artillerie l'ensemble des armes col...,208533468,208533787,None,missing_mention_with_sentence_modifications,[{'context': 'On appelle artillerie l'ensemble...,"Qin Shi Huang, ou Qin Shi Huangdi, fut d'abord..."
4,Artillerie,57485,Chu_ko_nu,None,[25],None,"J.-C.), dont une version améliorée pour tirer ...",chu ko nu,Histoire,None,On appelle artillerie l'ensemble des armes col...,208533468,208533787,None,missing_mention_with_sentence_modifications,[{'context': 'On appelle artillerie l'ensemble...,La chu ko nu ou arbalète à répétition est une ...


**Scoring the contexts**

In [10]:
data_dir = ready_to_score_data_dir
output_dir = scored_data_dir

subprocess.run([
    "python", "get_scores.py",
    "--data_dir", data_dir,
    "--output_dir", output_dir,
    "--langs", ",".join(langs),
    "--side_contexts_sizes", ",".join(map(str, side_context_sizes)),
    "--models_dir", "/dlabdata1/tsoares/models",
    "--models_prefix", "roberta_full",
    "--use_section_title",
    "--use_mentions",
    "--use_cuda",
    "--only_multilingual",
    "--multilingual_name", "multilingual-all"
])

scored_files_example = os.listdir(output_dir)[0]
scored_data = pd.read_parquet(f"{output_dir}/{scored_files_example}")

exemple = scored_data['all_contexts'][0][0]

for key in exemple.keys():
    print(f"{key}:\n{exemple[key]}\n")

Processing language fr
Processing context size 5


100%|██████████| 190/190 [29:39<00:00,  9.37s/it]


Processing language es
Processing context size 5


100%|██████████| 358/358 [56:01<00:00,  9.39s/it]


Processing language en
Processing context size 5


100%|██████████| 90/90 [14:09<00:00,  9.43s/it]


Processing language de
Processing context size 5


100%|██████████| 201/201 [31:29<00:00,  9.40s/it]


context:
On appelle artillerie l'ensemble des armes collectives ou lourdes — ou des troupes qui mettent en œuvre ces armes — servant à envoyer, à courte, moyenne ou grande distance, sur l'ennemi ou sur ses positions et ses équipements, divers projectiles de moyen ou gros calibre&nbsp;: obus, boulet, roquette, missile, pour appuyer ses propres troupes engagées dans une bataille terrestre, navale, antiaérienne ou un siège. Le terme serait apparu vers le XIIIe&nbsp;siècle, dérivant d’un mot du vieux français «&nbsp;artillier&nbsp;» qui servait à désigner les artisans fabricants d'armes et équipements de guerre. Ces artisans ont été pendant longtemps les seuls spécialistes dans le service de ces armes puisqu'ils les fabriquaient et les essayaient avant livraison. C'est pourquoi, jusqu'au XVIIIe&nbsp;siècle, ils étaient commissionnés par les souverains pour servir ces armes à la guerre. Canon exposé devant la basilique Notre-Dame de Montréal, Québec. Ainsi et par extension, le nom d'artille

**Colouring the pages**

In [11]:
output_dir = os.path.join(processed_data_dir, "colored_pages")

# Run the coloring script
subprocess.run([
    "python", "color_pages.py",
    "--scored_data_dir", scored_data_dir,
    "--html_dir", crawling_output_dir,
    "--files_link_output_dir", output_dir,
    "--langs", ",".join(langs),
    "--side_contexts_sizes", ",".join(map(str, side_context_sizes)),
    "--extra_crawled_files_dir", crawling_output_dir,
])

first_lang = langs[0]
first_context_size = str(side_context_sizes[0])
colored_files_paths = os.path.join(output_dir, first_lang, f"first_context_size.json")

print(f"Colored files path: {colored_files_paths}")



Processing language fr with side contexts size 5
Coloring with grey_to_red
Smoothing param: 1


  0%|          | 0/160 [00:00<?, ?it/s]

Style: highlight


100%|██████████| 160/160 [00:25<00:00,  6.33it/s]


Processing language es with side contexts size 5
Coloring with grey_to_red
Smoothing param: 1


...

Processing language de with side contexts size 5
Coloring with grey_to_red
Smoothing param: 1


  1%|          | 2/233 [00:00<00:12, 18.62it/s]

Style: highlight


100%|██████████| 233/233 [00:29<00:00,  7.95it/s]


Colored files path: /dlabdata1/boulenge/wikidumps_data/report_examples/colored_pages/fr/first_context_size.json
